# Fine-tune on CES Survey Data (QLoRA, Colab)

**Runtime → Change runtime type → GPU**

Edit the CONFIG cell below to change model, condition, and training params.

**To pull latest changes** after initial clone:
```python
!git -C article_silicon_sampling_quebec pull origin main
```

In [ ]:
# ============================================================
# CONFIG — edit these before running
# ============================================================

# Your HF token (https://huggingface.co/settings/tokens)
HF_TOKEN = ""  # <-- SET YOUR TOKEN HERE

# Your W&B API key (https://wandb.ai/authorize) — leave empty to disable
WANDB_API_KEY = ""  # <-- SET YOUR KEY HERE

# Model options (BASE models — not Instruct):
#   Qwen/Qwen2.5-0.5B                    (~3GB VRAM, RECOMMENDED for publication)
#   Qwen/Qwen2.5-1.5B                    (~5GB VRAM)
#   Qwen/Qwen2.5-3B                      (~8GB VRAM)
#   meta-llama/Llama-3.2-1B              (~4GB VRAM)
#   meta-llama/Llama-3.2-3B              (~8GB VRAM)
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Dataset condition (from huggingface.co/datasets/hubcad25/article_silicon_sampling_quebec_data)
# Options: finetune_train_4a.jsonl, 4b.jsonl, 5a.jsonl, 5b.jsonl
DATASET_FILE = "finetune_train_4a.jsonl"

# Training params
EPOCHS = 1          # 1 for quick eval, 3 for full run
MAX_TRAIN_SAMPLES = 30000  # Remove/comment for full dataset (~1h on T4)
HF_REPO = "hubcad25/qwen-0.5b-lora-4a-30k"  # Output HF repo
# ============================================================
import os
os.environ["HF_TOKEN"] = HF_TOKEN
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_FILE}")
print(f"Epochs: {EPOCHS}, max_train_samples: {MAX_TRAIN_SAMPLES or 'ALL'}")
print(f"W&B: {'enabled' if WANDB_API_KEY else 'disabled'}")

In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers accelerate peft datasets huggingface_hub wandb
!pip install -q polars

# Login to W&B if key is set
if WANDB_API_KEY:
    !wandb login $WANDB_API_KEY

In [ ]:
# Clone repo + download dataset
!git clone https://github.com/hubcad25/article_silicon_sampling_quebec.git
import os
os.chdir("article_silicon_sampling_quebec")

# Pull latest changes to get updated finetune.py with progress callback
!git -C article_silicon_sampling_quebec pull origin main

from huggingface_hub import hf_hub_download
dataset_local = hf_hub_download(
    repo_id="hubcad25/article_silicon_sampling_quebec_data",
    filename=DATASET_FILE,
    token=HF_TOKEN,
    repo_type="dataset",
)
print(f"Dataset: {dataset_local}")
!wc -l {dataset_local}

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
# Fix for Colab: ensure GPU is visible to subprocess
import os
if 'CUDA_VISIBLE_DEVICES' not in os.environ:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

In [ ]:
# Run fine-tuning
OUTPUT_DIR = "/content/model_output"
import subprocess

cmd = [
    "python", "scripts/finetune.py",
    "--data", dataset_local,
    "--model", MODEL_NAME,
    "--output_dir", OUTPUT_DIR,
    "--use_4bit",
    "--epochs", str(EPOCHS),
    "--batch_size", "4",
    "--grad_accum", "8",
    "--lr", "2e-4",
    "--max_seq_len", "2048",
    "--hf_repo", HF_REPO,
]
if MAX_TRAIN_SAMPLES:
    cmd.extend(["--max_train_samples", str(MAX_TRAIN_SAMPLES)])

if WANDB_API_KEY:
    cmd.append("--report_to")
    cmd.append("wandb")

print(f"Running: {' '.join(cmd[:8])} ...")
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
result = subprocess.run(cmd, env=env, capture_output=True, text=True)
if result.returncode == 0:
    print("Done. LoRA pushed to HF.")
else:
    print(f"Failed (exit {result.returncode})")
    print("=== STDERR ===")
    print(result.stderr[-4000:])
    print("=== STDOUT ===")
    print(result.stdout[-2000:])

---

## Monitoring (run while fine-tuning is running)

Run this cell in a **new cell** to monitor progress.

In [ ]:
# Monitor GPU usage while fine-tuning runs
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader

# Check for checkpoint files (shows if training is progressing)
import os, glob
if os.path.exists("/content/model_output"):
    checkpoints = sorted(glob.glob("/content/model_output/checkpoint-*"))
    print(f"Checkpoints: {len(checkpoints)}")
    if checkpoints:
        print(f"Latest: {checkpoints[-1]}")
else:
    print("No output yet — still initializing")